# Mirror ranking barplots from CSV

This notebook reads the already generated `framewise_target_class_counts.csv` file and produces the **symmetric mirror barplots** used for the ranking-consistency analysis.

## What it does
- loads the CSV produced by the framewise counting script
- selects the target classes (`person`, `car`)
- builds one **4-panel figure** per target class
- x-axis = frame rank sorted by clean count
- positive bars = normalized clean counts
- negative bars = normalized adverse counts
- exports the figures as **PDF**

No model inference is run here.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
# -----------------------
# USER CONFIG
# -----------------------
COUNTS_CSV = "./ranking_consistency_results/framewise_target_class_counts.csv"
SAVE_DIR = "./ranking_consistency_results"

TARGET_CLASSES = ["person", "car"]

PANELS = [
    ("our_dataset", "SegFormer-B5"),
    ("our_dataset", "Mask2Former-Swin-L"),
    ("acdc", "SegFormer-B5"),
    ("acdc", "Mask2Former-Swin-L"),
]

SAVE_PDF = True

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 10,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})


In [3]:
# -----------------------
# Helpers
# -----------------------
def spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) == 0 or len(y) == 0 or len(x) != len(y):
        return np.nan

    rx = pd.Series(x).rank(method="average").to_numpy()
    ry = pd.Series(y).rank(method="average").to_numpy()

    rx_mean = rx.mean()
    ry_mean = ry.mean()

    num = ((rx - rx_mean) * (ry - ry_mean)).sum()
    den = np.sqrt(((rx - rx_mean) ** 2).sum() * ((ry - ry_mean) ** 2).sum())

    if den < 1e-12:
        return np.nan
    return float(num / den)

def max_normalize(v: np.ndarray) -> np.ndarray:
    vmax = float(np.max(v)) if len(v) > 0 else 0.0
    if vmax <= 1e-12:
        return np.zeros_like(v, dtype=np.float64)
    return v / vmax


In [4]:
# -----------------------
# Main plotting function
# -----------------------
def save_mirror_ranking_figure(
    df: pd.DataFrame,
    target_class: str,
    save_dir: str,
    save_pdf: bool = True,
):
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=True)
    axes = axes.flatten()

    prepared = []
    global_max = 0.0

    for dataset_name, model_name in PANELS:
        df_m = df[
            (df["dataset_name"] == dataset_name) &
            (df["model"] == model_name) &
            (df["target_class"] == target_class)
        ].copy()

        if df_m.empty:
            prepared.append((dataset_name, model_name, None, np.nan))
            continue

        # Sort by clean count descending
        df_m = df_m.sort_values("clean_count", ascending=False).reset_index(drop=True)
        df_m["frame_rank"] = np.arange(len(df_m))

        clean_vals = df_m["clean_count"].to_numpy(dtype=np.float64)
        adv_vals = df_m["adv_count"].to_numpy(dtype=np.float64)

        clean_norm = max_normalize(clean_vals)
        adv_norm = max_normalize(adv_vals)

        df_m["clean_norm"] = clean_norm
        df_m["adv_norm"] = adv_norm

        rho = spearman_corr(clean_vals, adv_vals)

        local_max = max(
            float(clean_norm.max()) if len(clean_norm) > 0 else 0.0,
            float(adv_norm.max()) if len(adv_norm) > 0 else 0.0,
        )
        global_max = max(global_max, local_max)

        prepared.append((dataset_name, model_name, df_m, rho))

    y_lim = max(1.0, global_max * 1.15)

    for ax, (dataset_name, model_name, df_m, rho) in zip(axes, prepared):
        if df_m is None:
            ax.axis("off")
            continue

        x = df_m["frame_rank"].to_numpy()
        y_clean = df_m["clean_norm"].to_numpy()
        y_adv = -df_m["adv_norm"].to_numpy()

        ax.bar(x, y_clean, width=0.9, alpha=0.85, label="Clean")
        ax.bar(x, y_adv, width=0.9, alpha=0.85, label="Adverse")

        ax.axhline(0.0, color="black", linewidth=0.9)

        ax.set_title(f"{dataset_name} — {model_name}\nSpearman = {rho:.3f}", fontsize=13)
        ax.set_xlabel("Frame rank (ordered by clean count)")
        ax.set_ylim(-y_lim, y_lim)
        ax.set_xlim(-1, len(x))

        if len(x) > 20:
            step = max(1, len(x) // 10)
            ticks = np.arange(0, len(x), step)
            ax.set_xticks(ticks)

        ax.tick_params(axis="y", labelsize=11)
        ax.tick_params(axis="x", labelsize=10)

    axes[0].set_ylabel("Normalized target-class count", fontsize=14)
    axes[2].set_ylabel("Normalized target-class count", fontsize=14)

    handles, labels = axes[0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc="center left",
        bbox_to_anchor=(0.92, 0.5),
        frameon=False,
        fontsize=11,
    )

    fig.suptitle(f"Mirror ranking plot — target class: {target_class}", fontsize=16, y=0.98)
    fig.tight_layout(rect=[0, 0, 0.89, 0.96])

    plot_dir = os.path.join(save_dir, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    if save_pdf:
        out_path = os.path.join(plot_dir, f"mirror_ranking_{target_class}_4panels.pdf")
        fig.savefig(out_path, bbox_inches="tight")
    else:
        out_path = os.path.join(plot_dir, f"mirror_ranking_{target_class}_4panels.png")
        fig.savefig(out_path, dpi=200, bbox_inches="tight")

    plt.close(fig)
    print("Saved:", out_path)


In [5]:
# -----------------------
# Load CSV
# -----------------------
df = pd.read_csv(COUNTS_CSV)

required_cols = {
    "dataset_name",
    "model",
    "target_class",
    "pair_id",
    "clean_count",
    "adv_count",
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

df.head()


,dataset_name,model,target_class,pair_id,clean_count,adv_count,clean_path,adv_path
0,our_dataset,SegFormer-B5,person,10231074,0,0,/home/ace/Downloads/Dataset/Day/10231074_Day_E...,/home/ace/Downloads/Dataset/Night/10231074_Nig...
1,our_dataset,SegFormer-B5,person,108359,40,1329,/home/ace/Downloads/Dataset/Day/108359_Day_EXT...,/home/ace/Downloads/Dataset/Night/108359_Night...
2,our_dataset,SegFormer-B5,person,10901081,0,0,/home/ace/Downloads/Dataset/Day/10901081_Day_E...,/home/ace/Downloads/Dataset/Night/10901081_Nig...
3,our_dataset,SegFormer-B5,person,11101250,20807,22988,/home/ace/Downloads/Dataset/Day/11101250_Day_E...,/home/ace/Downloads/Dataset/Night/11101250_Nig...
4,our_dataset,SegFormer-B5,person,11131433,0,0,/home/ace/Downloads/Dataset/Day/11131433_Day_E...,/home/ace/Downloads/Dataset/Night/11131433_Nig...


In [6]:
# -----------------------
# Generate figures
# -----------------------
for target_class in TARGET_CLASSES:
    save_mirror_ranking_figure(
        df=df,
        target_class=target_class,
        save_dir=SAVE_DIR,
        save_pdf=SAVE_PDF,
    )


Saved: ./ranking_consistency_results/plots/mirror_ranking_person_4panels.pdf
Saved: ./ranking_consistency_results/plots/mirror_ranking_car_4panels.pdf


## Notes

- The CSV contains **raw counts**.
- The plots use **max-normalization** only for visualization.
- Spearman correlation is computed from the raw counts.
